# Local Reproduction: `morgan_2048_bce` Validation

This notebook reproduces validation for a fine-tuned `FingerprintHead` checkpoint using the same preprocessing and `SplittedDataModule` flow from `dreams/training/train.py`.

Expected result: validation AUROC close to the WandB value (~0.82).

In [1]:
from pathlib import Path

# Resolve project root as DreaMS/ (works when notebook lives in dreams-thesis-wa/notebooks)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "DreaMS":
    for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
        if p.name == "DreaMS":
            PROJECT_ROOT = p
            break

HDF5_PATH = PROJECT_ROOT / "dreams-thesis-wa/data/processed/MassSpecGym_splits/finetuning.hdf5"
CKPT_DIR = PROJECT_ROOT / "dreams-thesis-wa/results/model_runs/morgan_2048_bce/checkpoints"

assert HDF5_PATH.exists(), f"Missing finetuning.hdf5: {HDF5_PATH}"
assert CKPT_DIR.exists(), f"Missing checkpoint dir: {CKPT_DIR}"

ckpts = sorted(CKPT_DIR.glob("*.ckpt"))
assert ckpts, f"No checkpoint files in {CKPT_DIR}"
CKPT_PATH = ckpts[0]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("HDF5_PATH:", HDF5_PATH)
print("CKPT_PATH:", CKPT_PATH)

AssertionError: Missing checkpoint dir: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/model_runs/morgan_2048_bce/checkpoints

In [2]:
import pathlib
import torch
import pandas as pd

try:
    import lightning.pytorch as pl
except Exception:
    import pytorch_lightning as pl

from dreams.definitions import FOLD, PRETRAINED
import dreams.utils.data as du
from dreams.utils.dformats import DataFormatBuilder
from dreams.models.heads.heads import FingerprintHead

/Users/wouterachterberg/coding/DreaMS/.venv/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/Users/wouterachterberg/coding/DreaMS/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/User

In [3]:
# Train.py line 44-49 equivalent arguments
# preproc_precision = 64 if str(args.train_precision) == '64' else 32
train_precision = "32"
preproc_precision = 64 if str(train_precision) == "64" else 32

dformat = DataFormatBuilder("A").get_dformat()
prec_intens = 1.1
max_peaks_n = 100
spec_entropy_cleaning = False
mz_shift_aug_p = 0.0
mz_shift_aug_max = 0.0

spec_preproc = du.SpectrumPreprocessor(
    dformat=dformat,
    prec_intens=prec_intens,
    n_highest_peaks=max_peaks_n,
    spec_entropy_cleaning=spec_entropy_cleaning,
    precision=preproc_precision,
    mz_shift_aug_p=mz_shift_aug_p,
    mz_shift_aug_max=mz_shift_aug_max,
)

print("SpectrumPreprocessor created")
print("dformat:", type(dformat).__name__)
print("precision:", preproc_precision)
print("max_peaks_n:", max_peaks_n)

SpectrumPreprocessor created
dformat: DataFormatA
precision: 32
max_peaks_n: 100


In [ ]:
# Train.py line 78-95 equivalent data setup for .hdf5 fine-tuning
train_objective = "fp_morgan_2048"
batch_size = 256
num_workers_data = 0
n_samples = None
seed = 1

msdata = du.MSData(HDF5_PATH, in_mem=False)
dataset = msdata.to_torch_dataset(
    spec_preproc=spec_preproc,
    label=train_objective,
    dformat=dformat,
)

split_mask = pd.Series(msdata.get_values(FOLD)).apply(
    lambda x: x.decode("utf-8") if isinstance(x, bytes) else x
)

datamodule = du.SplittedDataModule(
    dataset=dataset,
    split_mask=split_mask,
    batch_size=batch_size,
    num_workers=num_workers_data,
    n_train_samples=n_samples,
    seed=seed,
)

# Ensure dataloaders are initialized before validate
if hasattr(datamodule, "setup"):
    try:
        datamodule.setup(stage="validate")
    except TypeError:
        datamodule.setup()



In [10]:
# Torch 2.6+ compatibility for checkpoints with non-tensor objects
try:
    torch.serialization.add_safe_globals([pathlib.PosixPath])
except Exception:
    pass

_original_torch_load = torch.load

def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _torch_load_compat
try:
    model = FingerprintHead.load_from_checkpoint(
        str(CKPT_PATH),
        backbone=PRETRAINED / "ssl_model.ckpt",
        map_location=torch.device("cpu"),
    )
finally:
    torch.load = _original_torch_load

model = model.eval().cpu()
print("Model loaded:", type(model).__name__)
print("Checkpoint:", CKPT_PATH.name)

Model loaded: FingerprintHead
Checkpoint: epoch=31-step=4224-val_loss=0.061447.ckpt


In [11]:
trainer = pl.Trainer(
    accelerator="cpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
)

val_metrics = trainer.validate(model=model, datamodule=datamodule)
print("Raw validate() output:")
print(val_metrics)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/wouterachterberg/coding/DreaMS/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/wouterachterberg/coding/DreaMS/.venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:442: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 10 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Validation: 0it [00:00, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        Val loss            0.06144613400101662
     ValBinaryAUROC         0.8994126319885254
    ValBinaryAccuracy       0.9845249056816101
  ValBinaryJaccardIndex      0.387276828289032
   ValBinaryPrecision       0.7379162907600403
     ValBinaryRecall        0.43807312846183777
   ValCosineSimilarity      0.6017806529998779
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Raw validate() output:
[{'ValBinaryJaccardIndex': 0.387276828289032, 'ValBinaryRecall': 0.43807312846183777, 'ValBinaryPrecision': 0.7379162907600403, 'ValBinaryAccuracy': 0.9845249056816101, 'ValBinaryAUROC': 0.8994126319885254, 'ValCosineSimilarity': 0.6017806

In [13]:
# Print AUROC-like keys and compare against expected ~0.82
metrics_dict = val_metrics[0] if isinstance(val_metrics, list) and val_metrics else {}

auroc_candidates = {k: v for k, v in metrics_dict.items() if "auroc" in k.lower()}
print("\nAUROC-related metrics:")
if auroc_candidates:
    for k, v in auroc_candidates.items():
        try:
            fv = float(v)
            print(f"- {k}: {fv:.6f}")
        except Exception:
            print(f"- {k}: {v}")
else:
    print("No AUROC key found in validate() output. Available keys:", list(metrics_dict.keys()))

expected = 0.89
tol = 0.05
for k, v in auroc_candidates.items():
    try:
        fv = float(v)
        ok = abs(fv - expected) <= tol
        print(f"\nSanity check for {k}: value={fv:.4f}, expected≈{expected:.2f}, within ±{tol:.2f}? {ok}")
    except Exception:
        pass


AUROC-related metrics:
- ValBinaryAUROC: 0.899413

Sanity check for ValBinaryAUROC: value=0.8994, expected≈0.89, within ±0.05? True
